## Interior point method

In [1]:
import numpy as np
import pandas as pd
import copy as copy
import scipy
import scipy.io
import time
import os
from scipy.linalg import solve, LinAlgWarning
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from matplotlib.animation import FuncAnimation, PillowWriter
import openpyxl
import xlsxwriter

from IPM_functions import (
    create_result_dataframes,
    update_result_dataframes,
    paso_intpoint,
    solve_catch_error,
    update_active_set_mask,
    active_set_diagnostics,
    progress_summary_df_clean,
    build_reduced_system,
    load_lp_problem,
    export_latex_tables,
    export_summary_to_latex
)

#from inpoint_methods import intpoint, intpointR, #,intpointR_mask

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

Matplotlib is building the font cache; this may take a moment.
/Users/fattybaking/Documents/thesis_int_point/IPM_functions.py:368: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
  .astype(str) + "\%"
/Users/fattybaking/Documents/thesis_int_point/IPM_functions.py:376: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
  .astype(str) + "\%"


# Método de puntos interiores con heurística (¿?)

In [ ]:
# Choose a file
#mat_file = 'lp_kb2.mat'     # b = 0
#mat_file = 'lp_afiro.mat'   # bmax = 500, and no mu tends to zero  and no condition problem
#mat_file = 'lp_blend.mat'    # bmax = 26.32, and no mu tends to zero.
#mat_file = 'lp_fit1d.mat'   # b = 0
mat_file = 'lp_fit1p.mat'   # bmax = 216, and no mu tends to zero, condition 10^12
#mat_file = 'lp_kb2.mat'   # bmax = 216, and no mu tends to zero, condition 10^12
#mat_file = 'lp_grow15.mat'  # b = 0, condition 10^6
#mat_file = 'lp_grow22.mat'  # b = 0, condition 1.4e7
#mat_file = 'lp_grow7.mat'   # b = 0, condition 1.2e7

# Load problem
Q, c, A, b, F, d, H = load_lp_problem(mat_file)
AT = A.T
FT = F.T

k = 0
n = Q.shape[0]
m = A.shape[0]
p = F.shape[0]

# Initial values
x = np.ones(n)
lamda = np.zeros(m)
mu = np.ones(p)
z = np.ones(p)
#z = F @ x - d + (0.5)*np.ones(p)

problem_info = {
    "problem_name": mat_file,
    "norm_inf_b": np.linalg.norm(b, np.inf),
    "dim_b": len(b),
    "num_zeros_b": np.sum(b==0),
    "dim_x": n,
    "dim_eq": m,
    "dim_ineq": p
}

tol = 1e-6
kmax = 100

sigma = 0.5 # Valor fijo
tau = sigma * np.dot(mu, z) / p

obj_function_df_value = 0.5 * x @ Q @ x + c @ x
max_complementarity_value = np.max(mu * z)

#Define and initialise the dataframes that will be used to record the results on every iteration for mu, z, tau, the maximum complementarity value: max(mu_i * z_i), and the objective value
mu_df, z_df, tau_df, obj_function_df, max_complementarity_df, active_set_history = create_result_dataframes(p)
mu_df, z_df, tau_df, obj_function_df, max_complementarity_df, active_set_history = update_result_dataframes(k, mu, z, tau, obj_function_df_value, max_complementarity_value, p, mu_df, z_df, tau_df, obj_function_df, max_complementarity_df, active_set_history)

regression_df = pd.DataFrame(columns=active_set_history.columns) #Store any regressed indexes for mu

r_x = Q @ x + AT @ lamda - FT @ mu + c
r_lamda = A @ x - b
r_mu = -F @ x + d + z
r_z = np.multiply(mu, z)

# ld1 is the right-hand side of the complete system with the residuals(but we use the reduces system to solve)
ld1 = np.concatenate((r_x, r_lamda, r_mu, r_z), 0)
norma_cnpo = np.linalg.norm(ld1,np.inf) # this is the CNPO (Condiciones necesarias de primer orden) without the perturbation

red_mu = []

trigger_for_reduced_loop = False
trigger_reason = None

while norma_cnpo > tol and k < kmax:
    # Update diagonal matrices Z and U inside the loop
    Z = np.diag(z)
    U = np.diag(mu)

    ### KKT Matrix (we don't actually use this for calculations as we're using the reduced system).
    # Initial residuals
    row1 = np.hstack((Q, AT, -FT, np.zeros((n, p))))
    row2 = np.hstack((A, np.zeros((m, m + p + p))))
    row3 = np.hstack((-F, np.zeros((p, m + p)), np.identity(p)))
    row4 = np.hstack((np.zeros((p, n + m)), Z, U))

    M = np.vstack((row1, row2, row3, row4))

    # Build the reduced system we will use for calculating
    D = np.diag(mu / z)
    G = Q+FT@D@F
    w = F @ x - d - (tau / mu)
        
    dg = Q @ x + AT @ lamda - FT@mu + c + FT@D@w
    
    # Define K as a block matrix
    m = A.shape[0]
    K = np.block([
        [G, AT],
        [A, np.zeros((m, m))]
    ])
    
    # Calculate the condition number of G
    condG = np.linalg.cond(G,1)
    
    # Define lado derecho of the system of equations (ld)
    ld = -np.concatenate([dg, A @ x - b])
    #norma_cnpo = np.linalg.norm(ld, np.inf)
    
    # Solve the linear system and catch any ill-conditioning warnings from SciPy
    delta_vector = solve_catch_error(K,ld,k) # Reduced system
    
    # Update the sections of the delta_vector
    delta_x     = delta_vector[0:n]
    delta_lamda = delta_vector[n:n + m]
    delta_mu    = - D @ (F @ delta_x + w)
    delta_z     = - ( (1 / mu) * (z * delta_mu - tau) + z )
    
    ### Step size reduction
    alpha_mu = paso_intpoint(mu, delta_mu)
    alpha_z  = paso_intpoint(z, delta_z)
    alpha    = min(alpha_mu, alpha_z)
    #alpha    = 0.995 * min(alpha_mu, alpha_z) # From Javi's thesis
    #print("alpha= " , alpha)
    
    # Percentage changes before we udpate mu and z
    mu_percentage_change = delta_mu/mu
    z_percentage_change  = delta_z/z
    
    # Update variables
    x += alpha * delta_x
    mu += alpha * delta_mu
    lamda += alpha * delta_lamda
    z += alpha * delta_z
    
    # Update tau and residuals
    tau = sigma * np.dot(mu, z) / (p)
    k += 1

    # Recalculate the objective function value and maximum complementarity value for this iteration
    obj_function_value = 0.5 * x @ Q @ x + c @ x
    max_complementarity_value = np.max(mu * z)

    #Update the dataframes that record the values for each iteration
    mu_df, z_df, tau_df, obj_function_df, max_complementarity_df, active_set_history = update_result_dataframes(k, mu, z, tau,
                             obj_function_value, max_complementarity_value, p,
                             mu_df, z_df, tau_df,
                             obj_function_df, max_complementarity_df,
                             active_set_history)

    # Review if mu and z have been meeting the criteria we want (for the last 3 iterations) in order to
    # eliminate them when we start our experiment.
    active_set_history = update_active_set_mask(mu, z, Q, k, tau, active_set_history, mu_df, mu_percentage_change, z_percentage_change)

    # Recalculate residuals
    r_x = Q @ x + AT @ lamda - FT @ mu + c
    r_lamda = A @ x - b
    r_mu = -F @ x + d + z
    r_z = np.multiply(mu, z)  # Element-wise product
    
    ld1 = np.concatenate((r_x, r_lamda, r_mu, r_z), 0)
    norma_cnpo = np.linalg.norm(ld1,np.inf)

    # We save these values again because we're saving them separately so we can update the original variables once we run the heuristic/experiment
    # This allows us to save relevant values of the regular IPM method.
    
    cnpo_norm = norma_cnpo
    primal_residual_norm   = np.linalg.norm(r_lamda, np.inf)
    inequality_residual_norm   = np.linalg.norm(r_mu, np.inf)
    max_complementarity   = np.max(r_z)
    barrier_parameter   = tau
    cond_G  = condG
    objective_value   =  0.5 * x @ Q @ x + c @ x

    #print("\niter=", k, "\t", "||cnpo||=", norma_cnpo)
    #print("Condition number of G:", np.linalg.cond(G,1))
    #print("rcond(G)", (1/np.linalg.cond(G,1)))
    #print("tau =",tau)
    #print(z)
    #print(mu)
    
    #------ This sections is used for reference, but not used for any calculations -------#

    #------ This checks how many of if all components of the complementarity are below a set tolerance (Courtesy of Andreas), and if indices are a subset of the previous indices

    # Create a boolean mask that checks if all components of the complementarity are below a set tolerance
    mask = mu*z <= 1e-5
    
    #print('cuantos chicos mu*z = %g, vector\n' % (sum(mask)), (mu*z)[mask])
    
    #If all values of mu*z are below the tolerance, we check if the indices that fulfil this are a subset of the previous indices that fulfilled this
    if all(mask): #Return True only if EVERY entry in mask is True
        # Once the mu and z have gotten sufficiently small,

        #neg_mu_mask = (-0.52 < mu_percentage_change) & (mu_percentage_change < -0.48)
        #const_z_mask = (-0.01 < z_percentage_change) & (z_percentage_change < 0.01)
        grow_z_mask = (z_percentage_change > -0.03) #& (z_percentage_change < 0.01)
        neg_mu = np.arange( len(mask) )[grow_z_mask]
        
        active_set_history = update_active_set_mask(mu, z, Q, k, tau, active_set_history, mu_df, mu_percentage_change, z_percentage_change)
        
        #print('mus chicos: vector\n', mu[neg_mu])
        if set(red_mu).issubset( neg_mu ):
            print ('IS subset: GOOD')
        else:
            print ('FAILS subset condition: BAD')
            
        #print('mus chicos: vector\n', neg_mu)
        #print('  change in percentages for mu \n', mu_percentage_change[neg_mu] )
        #print('zs tending to positive contants\n', z[neg_mu] )
        #print('  Largest and smallest change for percentages in entries of z  \n', min(z_percentage_change[neg_mu]), max(z_percentage_change[neg_mu] ))
        red_mu = neg_mu.copy()

    if np.all(mu * z < 1e-5):
        trigger_for_reduced_loop = True
        trigger_reason = "complementarity tolerance"
        print("Trigger: complementarity tolerance reached")
        break

    if condG > 1e15:
        trigger_for_reduced_loop = True
        trigger_reason = "ill-conditioned matrix"
        print("Trigger: ill-conditioned matrix detected")
        break
    
    # So here is when we start getting close to the issues of the matrix, as we already exited the while loop

print("Número de iteraciones hasta el criterio de paro:",k)

if not trigger_for_reduced_loop:
    print("Reduced system not triggered — skipping experiment.")

    problem_results_before_heuristic = {
    "KKT residual (‖r‖∞)": cnpo_norm,
    "Primal residual (‖r_p‖∞)": primal_residual_norm,
    "Inequality residual (‖r_d‖∞)": inequality_residual_norm,
    "Complementarity gap (max μᵢ zᵢ)": max_complementarity,
    "Barrier parameter (τ)": barrier_parameter,
    "Condition number κ(G)": cond_G,
    "Objective value f(x)": objective_value
}

    summary_before = progress_summary_df_clean(problem_results_before_heuristic)

    display(summary_before)

    export_summary_to_latex(summary_before, mat_file, output_dir="progress_summary")

if trigger_for_reduced_loop:
    print(f"Starting heuristic (triggered by {trigger_reason})")
    print(f"Condition number of G: {condG:.6e} ({condG:.6f})")
    print(f"rcond(G): {1/condG:.6e} ({1/condG:.6f})")

    k_heuristic = 0
    #tol_red = 1e-10 #check this
    kmax_heuristic = 10
    total_iterations= k + k_heuristic
    frozen_indices_mu_i_zero = set()
    stable_active_indices, regressed_indices = active_set_diagnostics(active_set_history,regression_df,p)
    frozen_indices_mu_i_zero.update(stable_active_indices)
    regressed_df = pd.DataFrame(columns=['Iteration', 'regressed_indices'])

    #Build system outside of the loop:
    Z = np.diag(z)
    U = np.diag(mu)

    # ────────── Build full system ──────────
    r1 = np.hstack((Q, AT, -FT @ U))
    r2 = np.hstack((A, np.zeros((m, m + p))))
    r3 = np.hstack((-U @ F, np.zeros((p, m)), -Z @ U))

    M = np.vstack((r1, r2, r3))   # full system
    cond_full = np.linalg.cond(M, 1)  

    #For recording the results with the new mu values once some turn zero
    mu_df0 = pd.DataFrame(columns=range(p))
    cmpdf0 = pd.DataFrame(columns=['cmp'])
    taudf0 = pd.DataFrame(columns=range(p))

    mu0 = mu.copy()
    mu0[stable_active_indices] = 0
    mu_df0.loc[total_iterations] = mu0

    red_results_df = pd.DataFrame(columns=['Función Objetivo', 'max(mu*z)', 'dim(M1)', 'Condición de G (full)', 'Condición de M1 (sistema nuevo)', 'Filas/Columnas eliminadas comparadas al original'])
    red_results_df = red_results_df.astype({
        'dim(M1)': 'int',
        'Filas/Columnas eliminadas comparadas al original': 'int'
    })
    red_results_df.loc[total_iterations] = [obj_function_value, max_complementarity_value, M.shape[0], cond_full, cond_full, len(stable_active_indices)]

    # this set contains the indices that will stay frozen once we determine that the components of mu are zero
    
    while np.linalg.norm(ld1, np.inf) > tol and k_heuristic < kmax_heuristic:
        print("Starting iteration #",total_iterations+1)
        Z = np.diag(z)
        U = np.diag(mu)

        # ────────── Build full system ──────────
        r1 = np.hstack((Q, AT, -FT @ U))
        r2 = np.hstack((A, np.zeros((m, m + p))))
        r3 = np.hstack((-U @ F, np.zeros((p, m)), -Z @ U))

        M = np.vstack((r1, r2, r3))   # full system

        # Build reduced system (as per my notes) which eliminates the ant
        M, M1, U1, ld1 = build_reduced_system(Q, AT, FT, U, A, F, Z, mu, x, lamda, c, b, d, tau, list(frozen_indices_mu_i_zero))

        cond_full = np.linalg.cond(M, 1)    # for full system
        cond_reduced = np.linalg.cond(M1, 1)  # for reduced system

        print("cond(M1, 1-norm):", np.linalg.cond(M1, 1))

        # SOLVE THE NEW SYSTEM AND EXTRACT THE CHANGES (deltas)
        delta_vector1 = scipy.linalg.solve(M1, -ld1)
        delta_x1     = delta_vector1[:n] # should be the same ?
        delta_lamda1 = delta_vector1[n:n + m] # should be the same ?
        Ddelta_mu1    = delta_vector1[n + m:]
        
        delta_mu1 = U1 @ Ddelta_mu1  # Divide each element of delta_mu1 by the corresponding diagonal element of U

        # Here's the vector transformed back into the original size problem
        full_delta_mu = np.zeros_like(mu)
        active_indices = [i for i in range(p) if i not in frozen_indices_mu_i_zero]
        for i, idx in enumerate(active_indices):
            full_delta_mu[idx] = delta_mu1[i]

        # Reconstruct Δz from the 3rd KKT row (primal feasibility)
        # δz = F δx - (Fx - d - z)  == F @ delta_x1 - r_mu  (since r_mu = -F@x + d + z)
        full_delta_z = F @ delta_x1 - (F @ x - d - z)

        # Recompute step sizes for the reduced-step directions **DOUBLE CHECK THIS LATER
        alpha_mu = paso_intpoint(mu, full_delta_mu)   # ensures μ + α·Δμ ≥ (1-τ)μ > 0
        alpha_z  = paso_intpoint(z,  full_delta_z)    # ensures z + α·Δz ≥ (1-τ)z > 0
        alpha    = min(alpha_mu, alpha_z)

        # Compute mu_percentage_change, z_percentage_change for the reduced step
        mu_percentage_change = full_delta_mu / mu
        z_percentage_change  = full_delta_z / z

        # Apply the step with the same step‑size alpha
        x      += alpha * delta_x1
        lamda  += alpha * delta_lamda1
        mu     += alpha * full_delta_mu
        z      += alpha * full_delta_z

        k_heuristic += 1
        total_iterations= k + k_heuristic

        # Update active_set_history after this reduced iteration
        active_set_history = update_active_set_mask(mu, z, Q, total_iterations, tau, active_set_history, mu_df, mu_percentage_change, z_percentage_change)

        # Recompute the highlighted columns for the next reduced iteration and see if any components stopped meeting the criteria, although it was meeting it before
        stable_active_indices, regressed_indices = active_set_diagnostics(active_set_history,regression_df,p)

        frozen_indices_mu_i_zero.update(stable_active_indices)

        mu0 = mu.copy()
        mu0[list(frozen_indices_mu_i_zero)] = 0
        mu_df0.loc[total_iterations] = mu0

        # Refresh τ and bump iteration counter (optional, for bookkeeping)
        tau = sigma * np.dot(mu, z) / p
        tau0 = sigma * np.dot(mu0, z) / p

        # Recompute residuals for diagnostics
        r_x = Q @ x + AT @ lamda - FT @ mu + c
        v10 = Q @ x + AT @ lamda - FT @ mu0 + c     # dual residual
        r_lamda = A @ x - b                             # primal residual
        r_mu = -F @ x + d + z                        # inequality residual
        r_z = mu * z 
        v40 = mu0 * z                               # complementarity product
        ld1 = np.concatenate((r_x, r_lamda, r_mu, r_z))
        ld10 = np.concatenate((v10, r_lamda, r_mu, v40))

        norm_after = np.linalg.norm(ld1, np.inf)
        norm_after0 = np.linalg.norm(ld10, np.inf)

        mu_df.loc[total_iterations] = mu    # dataframe for graphing the central path
        z_df.loc[total_iterations] = z  
        tau_df.loc[total_iterations] = np.full(p, tau)     # dataframe for graphing the central path
        
        obj_function_value = 0.5 * x @ Q @ x + c @ x
        max_complementarity_value = np.max(mu * z)
        cmp_val0 = np.max(mu0 * z)

        obj_function_df.loc[total_iterations] = obj_function_value
        max_complementarity_df.loc[total_iterations] = max_complementarity_value

        cmpdf0.loc[total_iterations] = cmp_val0
        taudf0.loc[total_iterations] = np.full(p, tau0)     # dataframe for graphing the central path

            # --- CHECK IF ALL μ ARE ZERO ---
        if np.all(mu0 == 0):
            print(f"All μ are zero at iteration {total_iterations}, skipping μ updates and ratio calculation")
            print(f"Final iteration reached at k_heuristic={k_heuristic}, all μ are zero.")
            tau0 = 0
            full_delta_mu = np.zeros_like(mu)
            mu_percentage_change = np.zeros_like(mu)
            k_red_final = k_heuristic
            k_heuristic = kmax_heuristic  # forces loop exit after this iteration
            print("Número de iteraciones del segundo loop:", k_red_final)
        else:
            print("Número de iteraciones del segundo loop:", k_heuristic)

        # After the reduced-loop iterations finish, right before computing problem_results_after_heuristic:
        D = np.diag(mu / z)      # updated D for the new mu and z
        D0 = np.diag(mu0 / z)      # updated D with mu0 and z 
        G = Q + FT @ D @ F 
        G0 = Q + FT @ D0 @ F       # updated G
        condG = np.linalg.cond(G, 1)
        condG0 = np.linalg.cond(G0, 1)

        red_results_df.loc[total_iterations] = [obj_function_value,cmp_val0,(M1.shape[0]),cond_full,np.linalg.cond(M1,1),(n+m+p-M1.shape[0])]
        #red_results_df0.loc[total_iterations] = [obj_function_value,cmp_val0,(M1.shape[0]),cond_full,np.linalg.cond(M1,1),(n+m+p-M1.shape[0])]

        # AFTER computing stable_active_indices, regressed_indices in the reduced loop:

        if len(regressed_indices) > 0:
            regressed_df = pd.concat([regressed_df,
                                    pd.DataFrame({'Iteration': [k_heuristic],
                                                    'regressed_indices': [regressed_indices]})],
                                    ignore_index=True)
            print(f"Iteration of reduced loop {k_heuristic}: Regressed columns:", regressed_indices)
        else:
            print(f"Iteration of reduced loop  {k_heuristic}: No regressed columns")


    problem_results_before_heuristic = {
    "KKT residual (‖r‖∞)": cnpo_norm,
    "Primal residual (‖r_p‖∞)": primal_residual_norm,
    "Inequality residual (‖r_d‖∞)": inequality_residual_norm,
    "Complementarity gap (max μᵢ zᵢ)": max_complementarity,
    "Barrier parameter (τ)": barrier_parameter,
    "Condition number κ(G)": cond_G,
    "Objective value f(x)": objective_value
    }
    
    problem_results_after_heuristic = {
        "KKT residual (‖r‖∞)": np.linalg.norm(ld10, np.inf),
        "Primal residual (‖r_p‖∞)": np.linalg.norm(r_lamda, np.inf),
        "Inequality residual (‖r_d‖∞)": np.linalg.norm(r_mu, np.inf),
        "Complementarity gap (max μᵢ zᵢ)": np.max(v40),
        "Barrier parameter (τ)": tau0,
        "Condition number κ(G)": condG0,  # updated condition number
        "Objective value f(x)": obj_function_value
    }


summary_df = progress_summary_df_clean(problem_results_before_heuristic, problem_results_after_heuristic)

display(summary_df)

export_latex_tables(summary_df, red_results_df, mat_file, p)

Loading problem from: lp_fit1p.mat
Norma infinita de b:  216.0
Problem loaded. n=1677, m=627, p=1677
Number of zeros in b: 0 (0.00%)
IS subset: GOOD
Trigger: complementarity tolerance reached
Número de iteraciones hasta el criterio de paro: 70
Starting heuristic (triggered by complementarity tolerance)
Condition number of G: 1.114122e+15 (1114121983931783.000000)
rcond(G): 8.975678e-16 (0.000000)
Consistently highlighted in last 5 iterations: []
How many in percentage of mu's dimension?  0.0 %
Starting iteration # 71
Deleted 0 rows/columns. M1 shape: (3981, 3981)
cond(M1, 1-norm): 31399449647.452297
Consistently highlighted in last 5 iterations: []
How many in percentage of mu's dimension?  0.0 %
Número de iteraciones del segundo loop: 1
Iteration of reduced loop  1: No regressed columns
Starting iteration # 72
Deleted 0 rows/columns. M1 shape: (3981, 3981)
cond(M1, 1-norm): 62735679582.59881
Consistently highlighted in last 5 iterations: []
How many in percentage of mu's dimension?  0

,Metric,Before,After,Did it decrease?
0,KKT residual (‖r‖∞),9.167518e-06,1.557035e-05,False
1,Primal residual (‖r_p‖∞),2.842171e-14,5.684342e-14,False
2,Inequality residual (‖r_d‖∞),1.421085e-14,2.273737e-13,False
3,Complementarity gap (max μᵢ zᵢ),9.167518e-06,5.436748e-07,True
4,Barrier parameter (τ),4.331742e-06,1.382013e-07,True
5,Condition number κ(G),1.114122e+15,3.564371e+16,False
6,Objective value f(x),1.364230e+05,1.364230e+05,True


In [3]:
mu_df

0             1          2           3              4     \
0   1.000000e+00      1.000000   1.000000    1.000000       1.000000   
1   7.455306e-01      1.008443   1.016297    1.009521       1.008323   
2   6.246359e-01      1.014884   1.024566    1.013552       1.014684   
3   3.870164e-01      1.031906   1.049773    1.021777       1.031506   
4   1.979318e-01      1.059397   1.105428    1.030766       1.058697   
5   1.627712e-01      1.073161   1.142002    1.041422       1.072304   
6   1.209553e-01      1.098524   1.219769    1.080738       1.097496   
7   6.071792e-02      1.160045   1.429888    1.192130       1.158489   
8   5.657796e-02      1.170446   1.483941    1.217283       1.168680   
9   3.924328e-02      1.220558   1.763508    1.333307       1.217690   
10  3.523251e-02      1.245392   1.965077    1.395565       1.241944   
11  2.779123e-02      1.304254   2.549684    1.551895       1.299311   
12  2.541141e-02      1.335393   3.083496    1.653760       1.329453   
13  2.517233e-02      1.339211   3.165372    1.665196       1.333060   
14  1.783228e-02      1.459083   5.795742    2.019579       1.445952   
15  1.348713e-02      1.607806  14.903077    2.953875       1.581815   
16  1.104160e-02      1.768019  20.775493    5.108439       1.722780   
17  9.168284e-03      1.971647  26.029013    9.917953       1.894686   
18  8.068084e-03      2.168858  28.936693   13.704656       2.052473   
19  4.129823e-03      3.182537  39.399886   23.584851       2.819484   
20  3.986520e-03      9.198143  39.513672   23.676483       3.187509   
21  3.984079e-03     23.291117  39.534160   23.691763       3.194557   
22  3.946827e-03    625.836095  40.360041   24.305916       3.299188   
23  3.945969e-03    637.960073  40.379478   24.319389       3.301966   
24  3.904397e-03   1288.457622  41.530589   25.080472       3.445200   
25  3.857536e-03   1946.299584  42.659544   25.870285       3.619175   
26  3.746025e-03   3419.463218  45.252610   27.775740       4.069592   
27  3.704868e-03   3939.188982  46.217438   28.646273       4.267183   
28  3.703402e-03   3961.095726  46.287174   28.727747       4.275756   
29  3.680569e-03   4323.910704  47.592670   30.328246       4.416039   
30  3.661722e-03   4557.389593  48.021242   30.644089       4.517690   
31  3.576308e-03   5607.213784  49.916902   31.938348       4.992109   
32  3.508524e-03   6429.455120  51.477805   32.703748       5.426587   
33  3.468716e-03   6926.926308  52.595026   32.723224       5.712187   
34  3.382477e-03   7924.423425  54.393592   34.212856       6.390987   
35  3.311275e-03   8743.772833  55.977427   35.672177       7.056360   
36  3.229119e-03   9686.291829  57.864143   37.492115       7.960957   
37  3.071617e-03  11437.081466  60.832509   39.912049      10.235743   
38  3.009606e-03  12221.648210  61.981761   41.105669      11.810171   
39  2.951287e-03  13042.132567  62.261558   41.157481      13.966977   
40  2.883992e-03  13927.272697  62.883011   41.245093      17.303146   
41  2.636762e-03  16384.867734  67.140310   44.123404      34.995039   
42  2.633489e-03  16418.762976  67.171617   44.176543      73.977397   
43  2.491040e-03  18732.566192  64.433775   50.596149    9835.999596   
44  2.461994e-03  19202.977434  63.866250   51.979573   11802.199100   
45  2.380785e-03  20601.223300  61.854322   56.714519   17735.247416   
46  2.338496e-03  21257.615449  61.095507   58.549470   20463.531742   
47  2.215248e-03  23157.789006  58.907924   63.865426   28351.696447   
48  2.089555e-03  25055.376502  56.799669   69.196580   36230.226777   
49  1.978760e-03  26713.154026  55.074799   73.777320   43025.626132   
50  1.928883e-03  27718.029735  54.559018   76.006052   46317.830309   
51  1.712966e-03  32025.468192  52.392929   85.488670   60360.203858   
52  1.614595e-03  33788.004790  51.448373   89.375249   66199.098949   
53  1.571221e-03  34562.673413  51.023554   90.976066   68913.645246   
54  1.429384e-03  36925.474562  49.848816   95.78324